# Findings 
| Model              | Test Accuracy | Macro F1 | Weighted F1 |
| ------------------ | ------------: | -------: | ----------: |
| SVM                |    **62.29%** | **0.24** |    **0.56** |
| ANN                |    **58.97%** | **0.19** |    **0.50** |
| Custom CNN         |    **64.47%** | **0.35** |    **0.60** |
| CNN + Augmentation |    **64.50%** | **0.34** |    **0.61** |
| MobileNetV2        |    **69.97%** | **0.45** |    **0.68** |


## What the results show
### SVM
Accuracy: 62.29%
Works reasonably on the dominant NV class.
Struggles heavily with minority classes.
Macro F1 of 0.24 shows that performance is uneven across classes.
### ANN
Lowest accuracy: 58.97%
Macro F1: 0.19
Struggles because flattened pixels don't preserve image structure.
### Custom CNN
Accuracy increased to 64.47%.
Macro F1 increased to 0.35.
CNN learns spatial image features better than ANN.
### CNN + Augmentation
Accuracy: 64.50%.
Very similar to the original CNN.
Augmentation helped some classes but did not produce a major overall improvement.
### MobileNetV2
Accuracy: 69.97%
Macro F1: 0.45
Weighted F1: 0.68
Better performance across several minority classes as well.

# 1. Traditional ML (SVM, KNN)
Load → Resize → Normalize → Flatten → Train/Test Split → Scale → SVM → Evaluate

In [3]:
# =========================
# Traditional ML - SVM
# =========================

import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Dataset paths
dataset_path = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input"
csv_path = os.path.join(dataset_path, "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv")
image_path = os.path.join(dataset_path, "ISIC_2019_Training_Input")

# Load labels
df = pd.read_csv(csv_path)

# Get class labels from the CSV
# Remove image ID and non-class columns
class_columns = [
    col for col in df.columns[1:]
    if col != "UNK"
]

labels = df[class_columns].values.argmax(axis=1)

print("Classes:", list(class_columns))
print("Number of classes:", len(class_columns))

# Image IDs
image_ids = df.iloc[:, 0].values

# Load and resize images
images = []

for image_id in image_ids:
    file_path = os.path.join(image_path, image_id + ".jpg")

    image = Image.open(file_path).convert("RGB")
    image = image.resize((32, 32))

    images.append(np.array(image))

# Convert to NumPy array
X = np.array(images)
y = labels

print("Images shape:", X.shape)
print("Labels shape:", y.shape)
print("Number of classes:", len(class_columns))

# Normalize pixel values
X = X / 255.0

# Flatten images for SVM
X = X.reshape(X.shape[0], -1)

print("Flattened shape:", X.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train SVM
svm_model = SVC(kernel="rbf", random_state=42)
svm_model.fit(X_train, y_train)

# Predictions
y_pred = svm_model.predict(X_test)

# Accuracy
print("\nSVM Test Accuracy:", accuracy_score(y_test, y_pred))

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=class_columns
))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

Classes: ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
Number of classes: 8
Images shape: (25331, 32, 32, 3)
Labels shape: (25331,)
Number of classes: 8
Flattened shape: (25331, 3072)

SVM Test Accuracy: 0.6228537596210776

Classification Report:
              precision    recall  f1-score   support

         MEL       0.63      0.35      0.45       904
          NV       0.66      0.93      0.77      2575
         BCC       0.47      0.60      0.53       665
          AK       1.00      0.01      0.01       173
         BKL       0.66      0.10      0.18       525
          DF       0.00      0.00      0.00        48
        VASC       0.00      0.00      0.00        51
         SCC       0.00      0.00      0.00       126

    accuracy                           0.62      5067
   macro avg       0.43      0.25      0.24      5067
weighted avg       0.61      0.62      0.56      5067


Confusion Matrix:
[[ 316  523   60    0    5    0    0    0]
 [  82 2388   97    0    8    0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 2. Vanilla ANN
Load → Resize → Normalize → Flatten → Train/Validation/Test Split → Scale → ANN → Evaluate

In [15]:
# Vanilla ANN - ISIC 2019

import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.utils import to_categorical


# 1. dataset load
dataset_path = "/kaggle/input/datasets/andrewmvd/isic-2019"

csv_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_GroundTruth.csv"
)

image_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_Input",
    "ISIC_2019_Training_Input"
)

df = pd.read_csv(csv_path)


# 2. Get class columns automatically
class_columns = [
    col for col in df.columns[1:]
    if col != "UNK"
]

labels = df[class_columns].values.argmax(axis=1)
image_ids = df.iloc[:, 0].values

print("Classes:", list(class_columns))
print("Number of classes:", len(class_columns))


# 3. Load and resize images
images = []

for image_id in image_ids:
    file_path = os.path.join(image_path, image_id + ".jpg")

    image = Image.open(file_path).convert("RGB")
    image = image.resize((32, 32))

    images.append(np.array(image))


X = np.array(images)
y = labels

print("Image shape:", X.shape)


# 4. Normalize pixel values
X = X / 255.0


# 5. Flatten images for ANN
X = X.reshape(X.shape[0], -1)

print("ANN input shape:", X.shape)


# 6. Train / validation / test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)


# 7. Scale features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


# 8. Convert labels to categorical
num_classes = len(class_columns)

y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)


# 9. Build ANN
model = Sequential([
    Input(shape=(X_train.shape[1],)),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(num_classes, activation="softmax")
])


# 10. Compile
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# 11. Train
history = model.fit(
    X_train,
    y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=10,
    batch_size=32,
    verbose=1
)


# 12. Evaluate
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test_cat,
    verbose=0
)

print("\nTest Accuracy:", test_accuracy)


# 13. Predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)


# 14. Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_columns,
        zero_division=0
    )
)


# 15. Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classes: ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
Number of classes: 8
Image shape: (25331, 32, 32, 3)
ANN input shape: (25331, 3072)


I0000 00:00:1790274400.344560      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790274400.347742      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10
 56/555 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3703 - loss: 2.8740

I0000 00:00:1790274406.131542     111 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.4860 - loss: 1.6059 - val_accuracy: 0.5361 - val_loss: 1.2290
Epoch 2/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5358 - loss: 1.3346 - val_accuracy: 0.5482 - val_loss: 1.2618
Epoch 3/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5387 - loss: 1.2831 - val_accuracy: 0.5550 - val_loss: 1.1678
Epoch 4/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5545 - loss: 1.2554 - val_accuracy: 0.5626 - val_loss: 1.1588
Epoch 5/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5638 - loss: 1.2281 - val_accuracy: 0.5624 - val_loss: 1.1504
Epoch 6/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5695 - loss: 1.2063 - val_accuracy: 0.5808 - val_loss: 1.1381
Epoch 7/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5729 - loss: 1.1998 - val_accuracy: 0.5821 - val_loss: 1.1506
Epoch 8/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5766 - loss: 1.1887 - val_accuracy: 0.5729 - val_

# 3. Custom CNN
Load → Resize → Normalize → Train/Validation/Test Split → CNN (Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → Dense) → Evaluate

In [17]:
# Custom CNN - ISIC 2019

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout


# 1. Load dataset
dataset_path = "/kaggle/input/datasets/andrewmvd/isic-2019"

csv_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_GroundTruth.csv"
)

image_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_Input",
    "ISIC_2019_Training_Input"
)

df = pd.read_csv(csv_path)


# 2. Get classes automatically
class_columns = [
    col for col in df.columns[1:]
    if col != "UNK"
]

labels = df[class_columns].values.argmax(axis=1)
image_ids = df.iloc[:, 0].values

num_classes = len(class_columns)

print("Classes:", list(class_columns))
print("Number of classes:", num_classes)


# 3. Load and resize images
images = []

for image_id in image_ids:
    file_path = os.path.join(image_path, image_id + ".jpg")

    image = Image.open(file_path).convert("RGB")
    image = image.resize((32, 32))

    images.append(np.array(image))

X = np.array(images)
y = labels

print("Image shape:", X.shape)


# 4. Normalize pixel values
X = X / 255.0


# 5. Train / validation / test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


# 6. Build CNN
model = Sequential([
    Input(shape=(32, 32, 3)),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(num_classes, activation="softmax")
])


# 7. Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# 8. Train
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


# 9. Evaluate
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", test_accuracy)


# 10. Predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)


# 11. Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_columns,
        zero_division=0
    )
)


# 12. Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classes: ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
Number of classes: 8
Image shape: (25331, 32, 32, 3)
Training: (17731, 32, 32, 3)
Validation: (3800, 32, 32, 3)
Testing: (3800, 32, 32, 3)
Epoch 1/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.5446 - loss: 1.2777 - val_accuracy: 0.5642 - val_loss: 1.1926
Epoch 2/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5733 - loss: 1.1901 - val_accuracy: 0.5787 - val_loss: 1.1505
Epoch 3/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5911 - loss: 1.1434 - val_accuracy: 0.5989 - val_loss: 1.0978
Epoch 4/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5996 - loss: 1.1075 - val_accuracy: 0.6126 - val_loss: 1.0454
Epoch 5/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6118 - loss: 1.0714 - val_accuracy: 0.6168 - val_loss: 1.0523
Epoch 6/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6246 - loss: 1.0403 - val_accuracy: 0.6118 - val_loss: 1.0254
Epoch 7/10
555/555 ━━━━━━━━━━

# 4. Custom CNN + Data Augmentation
Load → Resize → Normalize → Split → Data Augmentation → CNN → Train → Evaluate

In [1]:
# Custom CNN + Data Augmentation - ISIC 2019

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, RandomFlip, RandomRotation, RandomZoom
)


# 1. Load dataset
dataset_path = "/kaggle/input/datasets/andrewmvd/isic-2019"

csv_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_GroundTruth.csv"
)

image_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_Input",
    "ISIC_2019_Training_Input"
)

df = pd.read_csv(csv_path)


# 2. Get classes automatically
class_columns = [
    col for col in df.columns[1:]
    if col != "UNK"
]

labels = df[class_columns].values.argmax(axis=1)
image_ids = df.iloc[:, 0].values

num_classes = len(class_columns)

print("Classes:", list(class_columns))
print("Number of classes:", num_classes)


# 3. Load and resize images
images = []

for image_id in image_ids:
    file_path = os.path.join(image_path, image_id + ".jpg")

    image = Image.open(file_path).convert("RGB")
    image = image.resize((32, 32))

    images.append(np.array(image))

X = np.array(images)
y = labels

print("Image shape:", X.shape)


# 4. Normalize
X = X / 255.0


# 5. Train / validation / test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


# 6. Data augmentation
augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1)
])


# 7. Build CNN
model = Sequential([
    Input(shape=(32, 32, 3)),

    augmentation,

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(num_classes, activation="softmax")
])


# 8. Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# 9. Train
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


# 10. Evaluate
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", test_accuracy)


# 11. Predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)


# 12. Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_columns,
        zero_division=0
    )
)


# 13. Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classes: ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
Number of classes: 8
Image shape: (25331, 32, 32, 3)
Training: (17731, 32, 32, 3)
Validation: (3800, 32, 32, 3)
Testing: (3800, 32, 32, 3)


I0000 00:00:1790291336.020993      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790291336.023788      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5395 - loss: 1.2867 - val_accuracy: 0.5745 - val_loss: 1.1610
Epoch 2/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5706 - loss: 1.1975 - val_accuracy: 0.5876 - val_loss: 1.1195
Epoch 3/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5858 - loss: 1.1594 - val_accuracy: 0.6016 - val_loss: 1.1118
Epoch 4/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5937 - loss: 1.1276 - val_accuracy: 0.6166 - val_loss: 1.0724
Epoch 5/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6039 - loss: 1.1036 - val_accuracy: 0.6129 - val_loss: 1.0578
Epoch 6/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6076 - loss: 1.0824 - val_accuracy: 0.6163 - val_loss: 1.0452
Epoch 7/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6130 - loss: 1.0683 - val_accuracy: 0.6329 - val_loss: 1.0123
Epoch 8/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6194 - loss: 1.0509 - val_accuracy: 0.

# 5. Transfer Learning with MobileNetV2
Pretrained MobileNetV2 → Freeze base → Train classifier

In [2]:
# Transfer Learning - MobileNetV2

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


# 1. Load dataset
dataset_path = "/kaggle/input/datasets/andrewmvd/isic-2019"

csv_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_GroundTruth.csv"
)

image_path = os.path.join(
    dataset_path,
    "ISIC_2019_Training_Input",
    "ISIC_2019_Training_Input"
)

df = pd.read_csv(csv_path)


# 2. Get classes automatically
class_columns = [
    col for col in df.columns[1:]
    if col != "UNK"
]

labels = df[class_columns].values.argmax(axis=1)
image_ids = df.iloc[:, 0].values

num_classes = len(class_columns)

print("Classes:", list(class_columns))
print("Number of classes:", num_classes)


# 3. Load and resize images
images = []

for image_id in image_ids:
    file_path = os.path.join(image_path, image_id + ".jpg")

    image = Image.open(file_path).convert("RGB")
    image = image.resize((128, 128))

    images.append(np.array(image))

X = np.array(images)
y = labels

print("Image shape:", X.shape)


# 4. MobileNetV2 preprocessing
X = preprocess_input(X.astype("float32"))


# 5. Train / validation / test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


# 6. Load pretrained MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze pretrained layers
base_model.trainable = False


# 7. Build transfer learning model
model = Sequential([
    Input(shape=(128, 128, 3)),

    base_model,

    GlobalAveragePooling2D(),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(num_classes, activation="softmax")
])


# 8. Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# 9. Train classifier
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


# 10. Evaluate
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", test_accuracy)


# 11. Predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)


# 12. Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_columns,
        zero_division=0
    )
)


# 13. Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classes: ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
Number of classes: 8
Image shape: (25331, 128, 128, 3)
Training: (17731, 128, 128, 3)
Validation: (3800, 128, 128, 3)
Testing: (3800, 128, 128, 3)
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10


2026-09-24 23:28:34.675143: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 23:28:34.812155: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1790292517.497602     110 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


554/555 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5641 - loss: 1.2409

2026-09-24 23:28:52.559406: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 23:28:52.696095: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


555/555 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5641 - loss: 1.2406

2026-09-24 23:29:10.494406: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 23:29:10.644277: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 23:29:10.781147: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


555/555 ━━━━━━━━━━━━━━━━━━━━ 53s 63ms/step - accuracy: 0.6039 - loss: 1.1174 - val_accuracy: 0.6637 - val_loss: 0.9386
Epoch 2/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.6591 - loss: 0.9486 - val_accuracy: 0.6666 - val_loss: 0.9030
Epoch 3/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.6750 - loss: 0.8975 - val_accuracy: 0.6792 - val_loss: 0.8875
Epoch 4/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.6904 - loss: 0.8577 - val_accuracy: 0.6811 - val_loss: 0.8800
Epoch 5/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7031 - loss: 0.8111 - val_accuracy: 0.6839 - val_loss: 0.8778
Epoch 6/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7088 - loss: 0.7869 - val_accuracy: 0.6921 - val_loss: 0.8611
Epoch 7/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7245 - loss: 0.7476 - val_accuracy: 0.6974 - val_loss: 0.8550
Epoch 8/10
555/555 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7367 - loss: 0.7140 - val_accuracy: 0.69